In [1]:
# Python file for running the chi square values


##PARAMETES AND IMPORTS
# Pointing the path to the import and parameter files
import sys
sys.path.insert(0, '../../')
# Import list
from imports import *
# Parameter list
import param as pm

## DATE, OBSERVATION NAME, FREQUENCY SLICES, FOLDER FOR SAVING
fname, dtime=tools.timepoint(fname=pm.file, date=None)
fs_slice=pm.fs_slice
fe_slice=pm.fe_slice
ts_slice=pm.ts_slice
te_slice=pm.te_slice

Date of observation: 2019-02-25 02:40:11
Fname: 1551055211


In [2]:
##FREQUENCY
print ('Frequeny range: '+str(fs_slice)+'-'+str(fe_slice)+' MHz')

##TIME
print ('Time range: '+str(ts_slice)+'-'+str(te_slice)+' seconds')

##MASK
if pm.mask==False:
    print ('Mask: No')
else:
    print ('Mask: Yes')
    print ('Degree masking: '+str(pm.degree))
    
##DAMPNER
print ('Dampner value: ',pm.dampner)
if pm.dampner!=None:
    print ('Dampner sigma level: '+str(pm.dampner_sigma))
    

Frequeny range: 1100-1350 MHz
Time range: 775-1000 seconds
Mask: No
Dampner value:  goob
Dampner sigma level: 3


In [3]:
##INITIALIZING THE FUNCTION
sat = ss(file_name=fname,          
             sats_only=None, 
             data_loc=pm.data_save, 
             sat_loc=pm.data_save,
             survey_info=[pm.nd_s0, pm.nd_s0_coords, pm.frequency], 
             sat_info=pm.satellite_catalogue,
             plots_loc=pm.data_plot,
             sat_beam=pm.beam_model+'_beam_'+str(pm.fs)+'_'+str(pm.fe)+'MHz', 
             frequency_range=[pm.fs, pm.fe], 
             constellations=pm.constellations_remain,
             nearby_satellites=pm.nearby_constellations,
             verbose=False)

In [4]:
##ALPHA DICTIONARY - Check here for when there could be a length issue
# np.random.seed()
# alpha_dic = np.random.random(21)
alpha_dic = np.ones(21)

##SIGMA DICTIONARY - comes here
if pm.dampner!=None:
    sigma_dic = pm.dampner_sigma*np.ones(21)
else:
    sigma_dic =None

In [5]:
##EXCECUTING THE INITIAL FUNCTION
sat.excecute(a_param=alpha_dic,                                                # Should always check this value
                 obs_time_start=ts_slice, obs_time_end=te_slice, 
                 obs_frequency_start=fs_slice, obs_frequency_end=fe_slice, 
                 file_bias_choice=pm.bias, 
                 add_sub=[1, 1], 
                 attenuation_func=pm.dampner,
                 attenuation_sigma=sigma_dic, 
                 bandsize=None,
                 verbose=True)


Number of signals inside choice frequency range:  21


In [6]:
##CHI SQAURE METHOD
def chisq_func2(params, masking=pm.mask):
    
    a_param = params[:21]
    s_param = params[21:]
    
    
    """
    Chi2 function which will take in all the parameters for the satellites
    """
    # a_param = params
    # s_param = pm.dampner_sigma*np.ones(len(pd.read_csv(pm.satellite_catalogue)))
    sat.excecute(a_param=a_param, 
                 obs_time_start=ts_slice, obs_time_end=te_slice, 
                 obs_frequency_start=fs_slice, obs_frequency_end=fe_slice, 
                 file_bias_choice=pm.bias, 
                 add_sub=[1, 1], 
                 attenuation_func=pm.dampner,
                 attenuation_sigma=s_param, 
                 bandsize=None,
                 verbose=False)
    

##Masked
    if masking==True:
#         max_k = 100
#         zero_arr = np.zeros(sat.calibration_data_slice.shape)
#         mask_idx = np.where(sat.calibration_data_slice > max_k)

#         zero_arr[mask_idx]=1
#         simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=zero_arr.T)    #SIMULATIONS
#         data = np.ma.array(data=sat.calibration_data_slice.T, mask=zero_arr.T)  #DATA

        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=sat.mask_nearby_satellites_slice)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=sat.mask_nearby_satellites_slice)

##No mask
    else:
        simulation = np.ma.array(data=sat.simulation_TOD_slice.T, mask=None)
        data = np.ma.array(data=sat.calibration_data_slice.T, mask=None)

# ##Averaging over time - MISSING TIMER PARAMETER
#     data = tools.waterfall_avg_time(timer=pm.nd_s0, size=20, waterfall=data)
#     simulation = tools.waterfall_avg_time(timer=pm.nd_s0, size=20, waterfall=simulation)
    
    
    
    # sig = tools.radiometer_eq(data=data)  # Note this is sigma (expected noise level), it must be squared to give the radiometer equation
    sig = 1
    chi_num = simulation - data  # WANT THIS VALUE
    
    chi_sq = np.ma.sum(chi_num**2 / sig**2)
    
    print (chi_sq)
    return chi_sq       # Work on getting that chi_num value out


In [8]:
##BOUNDARY INFORMATION
bnd_val = (0., None)
bnds = [bnd_val for bnd_i in range(2*sat.alpha_len)]

In [9]:
param_in = 10*np.random.random(42)+1


In [10]:
param_in

array([ 4.13261687,  4.45313713,  1.43036706,  5.69032362,  2.36640426,
        3.43485689,  1.1197816 , 10.76913254,  8.58987201,  9.51643555,
        1.62197675,  1.87235747,  7.40256702,  9.91188038,  8.02874468,
        6.28025927,  2.39368238, 10.17714403,  4.24400213,  2.55275294,
        2.8743177 ,  1.83328721,  5.87694128,  6.04826053,  9.63221052,
        7.51395634,  7.86552574,  7.82160495,  1.89797696,  4.45066993,
        2.42931595,  6.23175226, 10.33885012,  2.89207186,  8.97516867,
        3.4820322 ,  8.63194929,  3.79949698,  3.37811825,  8.19351457,
        8.69670594,  6.4960636 ])

In [11]:
time_s = time.time()
##RUNNING THE OPTIMIZATION PARAMETERS
print ('Running optimization')
signal_PL = opt.minimize(fun=chisq_func2, 
                         x0=param_in,                           # Set to none becuase the number of signals will be determined by the bandsize 
                         method='Powell',
                         bounds=bnds, 
                         tol=1e-6, 
                         options={'maxiter':20})

time_e = time.time()


Running optimization
17012432.59167891
17012560.23240761
17012200.09482342
17011974.174135156
17011882.62745591
17012227.833365995
17011895.79438662
17011882.39907267
17011880.37850176
17011883.642138395
17011880.248443156
17011880.247512016
17011880.24725074
17011880.247250676
17011880.247250672
17011880.247424427
17011880.247250725
17011880.247250676
17011880.247250672
17010114.230298106
17016457.69513189
17007253.59658107
17005937.4531132
17006288.493191957
17005898.28140771
17005859.369412173
17005886.777789492
17005855.09330446
17005855.0056745
17005854.993571166
17005854.99349884
17005854.99349876
17005854.99349876
17005854.99349877
17005912.94799417
17015794.03708792
17003452.109797843
17003081.06100591
17003099.59313813
17003057.430660784
17003057.34761607
17003057.32108182
17003057.320918594
17003057.320918534
17003057.320918538
17003057.32091854
16349776.204094123
18209666.08120709
15078524.651953042
13873750.171698384
12712068.896203607
11692299.473501768
10913877.49055328
1

In [12]:
(time_e - time_s) / 60 /60

4.724871174759334

In [13]:
##RUNNING SECOND INITIALIZATION
print ('Running 2nd init')
sat2 =  ss(file_name=fname,          
             sats_only=None, 
             data_loc=pm.data_save, 
             sat_loc=pm.data_save,
             survey_info=[pm.nd_s0, pm.nd_s0_coords, pm.frequency], 
             sat_info=pm.satellite_catalogue,
             plots_loc=pm.data_plot,
             sat_beam=pm.beam_model+'_beam_'+str(pm.fs)+'_'+str(pm.fe)+'MHz', 
             frequency_range=[pm.fs,pm.fe], 
             constellations=pm.constellations_remain,
             nearby_satellites=pm.nearby_constellations,
             verbose=False)

sat2.excecute(a_param=signal_PL.x[:21],                  
                 obs_time_start=ts_slice, obs_time_end=te_slice, 
                 obs_frequency_start=fs_slice, obs_frequency_end=fe_slice, 
                 file_bias_choice=pm.bias, 
                 add_sub=[1, 1], 
                 attenuation_func=pm.dampner,
                 attenuation_sigma=signal_PL.x[21:], 
                 bandsize=None,
                 verbose=False)

Running 2nd init


In [14]:
##SAVING OUTPUTS TO FILE
data_info = {'initial':param_in,
             'time':[ts_slice, te_slice],
             'frequency_slice':[fs_slice, fe_slice],
             'best-fit':signal_PL.x,
             'chi2_value':signal_PL.fun,
             'chi2_div':signal_PL.fun/sat2.simulation_TOD_slice.size
}

In [15]:
a_param = data_info['best-fit'][:21]
s_param = data_info['best-fit'][21:]

In [16]:
a_param

array([1.04193329e+01, 2.60549675e+00, 9.37948836e-01, 2.91366599e-02,
       2.67716023e+00, 5.77645143e-01, 1.34520133e+00, 6.51319658e-01,
       3.35321954e-07, 1.76602474e+00, 1.09190747e+00, 3.49170489e+00,
       6.10118991e-07, 1.05814153e+00, 3.94820330e-01, 6.96102659e-01,
       2.39851275e+00, 3.58901381e-07, 2.01239581e+00, 1.67943709e+00,
       2.93713839e+00])

In [17]:
s_param

array([1.67454983e+01, 4.51077777e+06, 4.76364986e+00, 1.13570743e+06,
       1.58289880e+06, 2.34353521e+06, 1.61597168e+06, 2.05574712e+06,
       1.94944040e+07, 1.79477494e+06, 1.62044087e+06, 1.18045256e+02,
       4.38492800e-02, 1.61933627e+06, 2.05191547e+05, 1.61137695e+06,
       1.68920727e+06, 4.71043551e-01, 1.97066891e+06, 1.69377015e+02,
       1.91607125e-02])

In [18]:
data_info

{'initial': array([ 4.13261687,  4.45313713,  1.43036706,  5.69032362,  2.36640426,
         3.43485689,  1.1197816 , 10.76913254,  8.58987201,  9.51643555,
         1.62197675,  1.87235747,  7.40256702,  9.91188038,  8.02874468,
         6.28025927,  2.39368238, 10.17714403,  4.24400213,  2.55275294,
         2.8743177 ,  1.83328721,  5.87694128,  6.04826053,  9.63221052,
         7.51395634,  7.86552574,  7.82160495,  1.89797696,  4.45066993,
         2.42931595,  6.23175226, 10.33885012,  2.89207186,  8.97516867,
         3.4820322 ,  8.63194929,  3.79949698,  3.37811825,  8.19351457,
         8.69670594,  6.4960636 ]),
 'time': [775, 1000],
 'frequency_slice': [1100, 1350],
 'best-fit': array([1.04193329e+01, 2.60549675e+00, 9.37948836e-01, 2.91366599e-02,
        2.67716023e+00, 5.77645143e-01, 1.34520133e+00, 6.51319658e-01,
        3.35321954e-07, 1.76602474e+00, 1.09190747e+00, 3.49170489e+00,
        6.10118991e-07, 1.05814153e+00, 3.94820330e-01, 6.96102659e-01,
        2.398

In [ ]:
pickle.dump(
    data_info, open(
        pm.data_save+pm.folder+'full_'+fname+'_mask_satellite_f_'+str(pm.fs)+'_'+str(pm.fe)\
        +'MHz_'+str(fs_slice)+'_'+str(fe_slice)+\
        'MHz'+'_'+str(ts_slice)+'_'+str(te_slice)+'_sec_free_atten_mixed_sig1'+pm.save_suffix+'.p', 'wb'))
